# Handle Class Imbalance with SMOTENC

This notebook applies SMOTENC only to the training split after the scaled dataset has been created. The continuous columns stay numeric, while the binary, binned, and one-hot encoded columns are treated as categorical features during resampling.

In [2]:
from pathlib import Path

import pandas as pd
from imblearn.over_sampling import SMOTENC
from sklearn.model_selection import train_test_split

BASE_DIR = Path(r'c:/Users/2021ICTS28/Desktop/end-to-end-credit-card-fraud-detection-system')
INPUT_PATH = BASE_DIR / 'dataset' / 'processed' / 'credit_card_fraud_scaled.csv'
OUTPUT_PATH = BASE_DIR / 'dataset' / 'processed' / 'credit_card_fraud_smotenc_train.csv'
TARGET_COL = 'is_fraud'
ID_COL = 'transaction_id'
CONTINUOUS_COLS = ['amount', 'transaction_hour', 'velocity_last_24h', 'cardholder_age']

In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f'Input file not found: {INPUT_PATH}. Run handle_scaling.ipynb first.'
    )

df = pd.read_csv(INPUT_PATH)
print(f'Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns')

if ID_COL in df.columns:
    df = df.drop(columns=[ID_COL])
    print(f'Dropped {ID_COL}')

if TARGET_COL not in df.columns:
    raise ValueError(f'Missing target column: {TARGET_COL}')

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print('Target distribution before split:')
print(y.value_counts().sort_index())

Loaded dataset: 10,000 rows x 13 columns
Target distribution before split:
is_fraud
0    9849
1     151
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

categorical_cols = [col for col in X_train.columns if col not in CONTINUOUS_COLS]
categorical_features = [X_train.columns.get_loc(col) for col in categorical_cols]

if not categorical_features:
    raise ValueError('No categorical columns were detected for SMOTENC.')

print(f'Continuous columns: {CONTINUOUS_COLS}')
print(f'Categorical columns: {categorical_cols}')
print(f'Train shape before SMOTENC: {X_train.shape[0]:,} rows x {X_train.shape[1]:,} columns')
print(f'Test shape: {X_test.shape[0]:,} rows x {X_test.shape[1]:,} columns')

Continuous columns: ['amount', 'transaction_hour', 'velocity_last_24h', 'cardholder_age']
Categorical columns: ['foreign_transaction', 'location_mismatch', 'device_trust_score_binned', 'merchant_category_Clothing', 'merchant_category_Electronics', 'merchant_category_Food', 'merchant_category_Grocery', 'merchant_category_Travel']
Train shape before SMOTENC: 8,000 rows x 12 columns
Test shape: 2,000 rows x 12 columns


In [5]:
df.head()

,amount,transaction_hour,foreign_transaction,location_mismatch,velocity_last_24h,cardholder_age,is_fraud,device_trust_score_binned,merchant_category_Clothing,merchant_category_Electronics,merchant_category_Food,merchant_category_Grocery,merchant_category_Travel
0,-0.560847,1.503345,0,0,0.706613,-0.231580,0,2,0,1,0,0,0
1,2.446026,-1.241383,1,0,-0.710297,1.370727,0,3,0,0,0,0,1
2,0.469007,0.781048,0,0,-0.710297,1.170439,0,1,0,0,0,1,0
3,-0.021683,-1.096923,0,1,0.706613,-0.632157,0,2,0,0,0,1,0
4,-0.925016,0.492130,0,0,-1.418753,0.035471,0,3,0,0,1,0,0


In [6]:
smote = SMOTENC(categorical_features=categorical_features, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print('Target distribution after SMOTENC:')
print(pd.Series(y_train_resampled).value_counts().sort_index())

train_resampled = X_train_resampled.copy()
train_resampled[TARGET_COL] = y_train_resampled
train_resampled.to_csv(OUTPUT_PATH, index=False)

print(f'Saved resampled training data to: {OUTPUT_PATH}')
print(f'Resampled training shape: {train_resampled.shape[0]:,} rows x {train_resampled.shape[1]:,} columns')

Target distribution after SMOTENC:
is_fraud
0    7879
1    7879
Name: count, dtype: int64
Saved resampled training data to: c:\Users\2021ICTS28\Desktop\end-to-end-credit-card-fraud-detection-system\dataset\processed\credit_card_fraud_smotenc_train.csv
Resampled training shape: 15,758 rows x 13 columns
